# Exercise 4 Solution: Advanced Analytics

Complete solutions for window functions, CTEs, and visualizations.

In [ ]:
import duckdb
import pandas as pd
import plotly.express as px

conn = duckdb.connect()

## Setup: Load Data

In [ ]:
# Load all required tables
conn.execute("""
    CREATE TABLE products AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.Product.csv');
    CREATE TABLE product_categories AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.ProductCategory.csv');
    CREATE TABLE product_subcategories AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.ProductSubcategory.csv');
    CREATE TABLE sales_orders AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesOrderHeader.csv');
    CREATE TABLE sales_details AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesOrderDetail.csv');
    CREATE TABLE territories AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesTerritory.csv');
""")

print("✓ All tables loaded")

## Task 1: Window Functions - Ranking

a) Create a ranking of products by list price within each category.

**Hint:** Use `ROW_NUMBER()` or `RANK()`

In [ ]:
# Solution
conn.execute("""
    SELECT 
        pc.Name AS category_name,
        p.Name AS product_name,
        p.ListPrice,
        ROW_NUMBER() OVER (PARTITION BY pc.Name ORDER BY p.ListPrice DESC) AS price_rank
    FROM products p
    JOIN product_subcategories ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
    JOIN product_categories pc ON ps.ProductCategoryID = pc.ProductCategoryID
    WHERE p.ListPrice > 0
    ORDER BY pc.Name, price_rank
""").df()

b) Show only the top 3 most expensive products per category

In [ ]:
# Solution
conn.execute("""
    WITH ranked AS (
        SELECT 
            pc.Name AS category_name,
            p.Name AS product_name,
            p.ListPrice,
            ROW_NUMBER() OVER (PARTITION BY pc.Name ORDER BY p.ListPrice DESC) AS rn
        FROM products p
        JOIN product_subcategories ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
        JOIN product_categories pc ON ps.ProductCategoryID = pc.ProductCategoryID
        WHERE p.ListPrice > 0
    )
    SELECT category_name, product_name, ListPrice, rn AS rank
    FROM ranked
    WHERE rn <= 3
    ORDER BY category_name, rn
""").df()

## Task 2: Running Totals

Calculate the running sum of sales (TotalDue) over time (OrderDate) for each territory.

**Hint:** Use `SUM() OVER (PARTITION BY ... ORDER BY ...)`

In [ ]:
# Solution
conn.execute("""
    SELECT 
        t.Name AS territory_name,
        so.OrderDate::DATE AS order_date,
        so.TotalDue,
        SUM(so.TotalDue) OVER (
            PARTITION BY t.Name 
            ORDER BY so.OrderDate
        ) AS running_total
    FROM sales_orders so
    JOIN territories t ON so.TerritoryID = t.TerritoryID
    ORDER BY t.Name, so.OrderDate
    LIMIT 50
""").df()

## Task 3: Moving Averages

Calculate the 7-day moving average of daily sales.

**Hint:** Use `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW`

In [ ]:
# Solution
conn.execute("""
    WITH daily_sales AS (
        SELECT 
            OrderDate::DATE AS sale_date,
            SUM(TotalDue) AS daily_total
        FROM sales_orders
        GROUP BY OrderDate::DATE
    )
    SELECT 
        sale_date,
        daily_total,
        ROUND(AVG(daily_total) OVER (
            ORDER BY sale_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 2) AS moving_avg_7d
    FROM daily_sales
    ORDER BY sale_date
""").df()

## Task 4: CTEs (Common Table Expressions)

Create a CTE-based query that:
1. Joins all sales with product information (CTE: `sales_with_products`)
2. Aggregates by category (CTE: `category_sales`)
3. Calculates each category's share of total revenue

In [ ]:
# Solution
conn.execute("""
    WITH sales_with_products AS (
        SELECT 
            sd.SalesOrderID,
            sd.LineTotal,
            pc.Name AS category_name
        FROM sales_details sd
        JOIN products p ON sd.ProductID = p.ProductID
        JOIN product_subcategories ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
        JOIN product_categories pc ON ps.ProductCategoryID = pc.ProductCategoryID
    ),
    category_sales AS (
        SELECT 
            category_name,
            SUM(LineTotal) AS category_revenue
        FROM sales_with_products
        GROUP BY category_name
    ),
    total_revenue AS (
        SELECT SUM(category_revenue) AS grand_total FROM category_sales
    )
    SELECT 
        cs.category_name,
        cs.category_revenue,
        ROUND(100.0 * cs.category_revenue / tr.grand_total, 2) AS revenue_share_pct
    FROM category_sales cs
    CROSS JOIN total_revenue tr
    ORDER BY category_revenue DESC
""").df()

## Task 5: Cohort Analysis

Analyze purchasing behavior by month:
- Group customers by their first purchase month (cohort)
- Show for each cohort how many customers purchased again in subsequent months

**Hint:** This is an advanced task!

In [ ]:
# Solution
conn.execute("""
    WITH customer_first_order AS (
        SELECT 
            CustomerID,
            DATE_TRUNC('month', MIN(OrderDate)) AS cohort_month
        FROM sales_orders
        GROUP BY CustomerID
    ),
    customer_orders AS (
        SELECT 
            so.CustomerID,
            DATE_TRUNC('month', so.OrderDate) AS order_month,
            cfo.cohort_month
        FROM sales_orders so
        JOIN customer_first_order cfo ON so.CustomerID = cfo.CustomerID
    )
    SELECT 
        cohort_month,
        DATEDIFF('month', cohort_month, order_month) AS months_since_first,
        COUNT(DISTINCT CustomerID) AS customer_count
    FROM customer_orders
    GROUP BY cohort_month, months_since_first
    HAVING months_since_first <= 12
    ORDER BY cohort_month, months_since_first
    LIMIT 50
""").df()

## Task 6: Visualization

a) Create a bar chart of top 10 products by revenue

In [ ]:
# Solution
df = conn.execute("""
    SELECT p.Name AS product_name, SUM(sd.LineTotal) AS total_revenue
    FROM sales_details sd
    JOIN products p ON sd.ProductID = p.ProductID
    GROUP BY p.Name
    ORDER BY total_revenue DESC
    LIMIT 10
""").df()

fig = px.bar(df, x='product_name', y='total_revenue', 
             title='Top 10 Products by Revenue')
fig.update_xaxes(tickangle=45)
fig.show()

b) Create a line chart showing monthly sales trends

In [ ]:
# Solution
df_monthly = conn.execute("""
    SELECT DATE_TRUNC('month', OrderDate)::DATE AS month, SUM(TotalDue) AS revenue
    FROM sales_orders
    GROUP BY 1 ORDER BY 1
""").df()

fig = px.line(df_monthly, x='month', y='revenue', title='Monthly Sales Trends')
fig.show()

## Task 7: Pivot Table

Create a pivot table with:
- Rows: Product categories
- Columns: Years
- Values: Total revenue

**Hint:** Use `PIVOT` or `CASE WHEN`

In [ ]:
# Solution
conn.execute("""
    SELECT 
        pc.Name AS category,
        SUM(CASE WHEN YEAR(so.OrderDate) = 2011 THEN sd.LineTotal ELSE 0 END) AS \"2011\",
        SUM(CASE WHEN YEAR(so.OrderDate) = 2012 THEN sd.LineTotal ELSE 0 END) AS \"2012\",
        SUM(CASE WHEN YEAR(so.OrderDate) = 2013 THEN sd.LineTotal ELSE 0 END) AS \"2013\",
        SUM(CASE WHEN YEAR(so.OrderDate) = 2014 THEN sd.LineTotal ELSE 0 END) AS \"2014\"
    FROM sales_details sd
    JOIN sales_orders so ON sd.SalesOrderID = so.SalesOrderID
    JOIN products p ON sd.ProductID = p.ProductID
    JOIN product_subcategories ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
    JOIN product_categories pc ON ps.ProductCategoryID = pc.ProductCategoryID
    GROUP BY pc.Name
    ORDER BY pc.Name
""").df()

## Bonus Task: RFM Analysis

Perform an RFM analysis (Recency, Frequency, Monetary):
- **Recency:** How long ago was the last purchase?
- **Frequency:** How often has the customer purchased?
- **Monetary:** How much has the customer spent?

Segment customers into 5 RFM groups (1 = worst, 5 = best).

In [ ]:
# Solution
conn.execute("""
    WITH customer_rfm AS (
        SELECT 
            CustomerID,
            DATEDIFF('day', MAX(OrderDate), (SELECT MAX(OrderDate) FROM sales_orders)) AS recency_days,
            COUNT(*) AS frequency,
            SUM(TotalDue) AS monetary
        FROM sales_orders
        GROUP BY CustomerID
    ),
    rfm_scores AS (
        SELECT 
            CustomerID,
            recency_days, frequency, monetary,
            NTILE(5) OVER (ORDER BY recency_days DESC) AS R_score,
            NTILE(5) OVER (ORDER BY frequency) AS F_score,
            NTILE(5) OVER (ORDER BY monetary) AS M_score
        FROM customer_rfm
    )
    SELECT 
        CustomerID, recency_days, frequency, ROUND(monetary, 2) AS monetary,
        R_score, F_score, M_score,
        R_score + F_score + M_score AS RFM_total,
        CASE 
            WHEN R_score + F_score + M_score >= 13 THEN 'Champions'
            WHEN R_score + F_score + M_score >= 10 THEN 'Loyal'
            WHEN R_score + F_score + M_score >= 7 THEN 'Potential'
            ELSE 'At Risk'
        END AS segment
    FROM rfm_scores
    ORDER BY RFM_total DESC
    LIMIT 20
""").df()

## 🎉 Congratulations!

You can now perform advanced analytics with DuckDB!